# Ishwarambare App - Database Testing Notebook

This notebook is for testing and exploring the PostgreSQL database connection and operations.

In [ ]:
# Import required libraries
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

# Load environment variables
load_dotenv()

# Database connection
DATABASE_URL = os.getenv("DATABASE_URL")
print(f"📊 Database URL: {DATABASE_URL}")

In [ ]:
# Test database connection
engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user, version()"))
    row = result.fetchone()
    
    print("✅ Database connection successful!")
    print(f"   Database: {row[0]}")
    print(f"   User: {row[1]}")
    print(f"   Version: {row[2]}")

In [ ]:
# List all tables in the database
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public'
        ORDER BY table_name
    """))
    tables = [row[0] for row in result.fetchall()]
    
    if tables:
        print(f"📋 Tables in database ({len(tables)}):")
        for table in tables:
            print(f"   - {table}")
    else:
        print("📋 No tables found yet (run the app to create them)")

In [ ]:
# Query portfolio data (if tables exist)
try:
    df = pd.read_sql("SELECT * FROM portfolios", engine)
    print(f"📊 Portfolios ({len(df)} records):")
    display(df.head())
except Exception as e:
    print(f"⚠️  Could not query portfolios: {e}")

In [ ]:
# Query alerts data (if tables exist)
try:
    df = pd.read_sql("SELECT * FROM alerts ORDER BY created_at DESC LIMIT 10", engine)
    print(f"📊 Recent Alerts:")
    display(df)
except Exception as e:
    print(f"⚠️  Could not query alerts: {e}")

In [ ]:
# Database statistics
with engine.connect() as conn:
    print("📈 Database Statistics:\n")
    
    tables = ['portfolios', 'alerts', 'articles', 'items']
    
    for table in tables:
        try:
            result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
            count = result.scalar()
            print(f"  {table}: {count} records")
        except Exception as e:
            print(f"  {table}: Table not found")

## Custom Queries

Use the cell below to run your own SQL queries:

In [ ]:
# Run custom SQL query
with engine.connect() as conn:
    result = conn.execute(text("""
        -- Your SQL query here
        SELECT current_timestamp as now
    """))
    
    df = pd.DataFrame(result.fetchall(), columns=result.keys())
    display(df)